# Download from Dataverse 
Our goal is to access ioerDATA via Dataverse API.
In this chapter we'll:
  1. Search the dataset using the DOI
  2. Examine dataset contents (metadata/files)
  3. Download the dataset
  4. Load and visualize data

# Dependencies and Base Configuration

In [2]:
import requests
import zipfile, io
from pathlib import Path

# Base URL and DOI for ioerDATA
BASE = "https://data.fdz.ioer.de"
DOI = "10.71830/AFW3N3"

#  Search for the dataset using the Search API
This code blocks searches dataset from IOER dataverse using api

In [ ]:
search_url = f"{BASE}/api/search?q={DOI}&type=dataset"
r = requests.get(search_url)
r.raise_for_status()
search_json = r.json()
search_json


In [5]:
def search_dataset_unique(doi, base=BASE):
    search_url = f"{base}/api/search"
    params = {
        "q": doi,
        "type": "dataset",
        "per_page": 100  # increase page size if many hits
    }
    resp = requests.get(search_url, params=params)
    resp.raise_for_status()
    j = resp.json()
    
    items = j.get("data", {}).get("items", [])
    unique = {}
    for item in items:
        pid = item.get("persistentId")
        if pid is None:
            pid = item.get("id")
        # Only keep the first encountered item per pid
        if pid not in unique:
            unique[pid] = item
    
    # Return deduped list
    return list(unique.values())

# Test it
unique_hits = search_dataset_unique(DOI)
print(f"Found {len(unique_hits)} unique dataset(s):")
for hit in unique_hits:
    print(" •", hit.get("name"), hit.get("persistentId"), "id:", hit.get("id"))

Found 1 unique dataset(s):
 • Replication Package for: Climate Regulation in Cities None id: None


# This returns dataset metadata (publication date, title, etc.). We'll extract the persistentId to compare our dataset.

In [24]:
items = search_json["data"]["items"]
dataset_hits = [it for it in items if it.get("type") == "dataset"]
assert dataset_hits, "No dataset hits returned"
first = dataset_hits[0]
first["global_id"]  # e.g. 'doi:10.xxxxx/YYY'
first["name"], first["url"]

('Replication Package for: Climate Regulation in Cities',
 'https://doi.org/10.71830/AFW3N3')

We download each file individually instead of the entire bundle to avoid server-side issues. The code handles original formats and keeps track of any errors.

In [26]:
from pathlib import Path

out_dir = Path("../../00_data/downloaded_data")
out_dir.mkdir(exist_ok=True)

def download_file(url, params, dest):
    resp = requests.get(url, params=params, stream=True)
    resp.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=1_048_576):
            f.write(chunk)

errors = []
for f in files:
    label = f["label"]
    file_id = f["dataFile"]["id"]
    dest = out_dir / label
    try:
        download_file(f"{BASE}/api/access/datafile/{file_id}", {"format": "original"}, dest)
    except Exception as e:
        errors.append((label, str(e)))

print("Downloaded files to:", out_dir)
if errors:
    print("Download errors:", errors)

Downloaded files to: ../../00_data/downloaded_data
Download errors: [('climate_regulation_in_cities.gpkg', '403 Client Error: Forbidden for url: https://data.fdz.ioer.de/api/access/datafile/901?format=original')]
